To run this notebook, please select L4 GPU

In [1]:
!pip install vllm

# After run a restart window will pop up, please click restart

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cuda-tile[tileiras] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 138.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 

In [1]:
!git clone https://github.com/rednote-hilab/dots.mocr.git
%cd dots.mocr

Cloning into 'dots.mocr'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 100 (delta 4), reused 9 (delta 3), pack-reused 89 (from 1)
Receiving objects: 100% (100/100), 52.96 MiB | 12.25 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/dots.mocr


In [2]:
!pip install -e .
%cd ..

Obtaining file:///content/dots.mocr
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 1.3.1
    Uninstalling starlette-1.3.1:
      Successfu

In [3]:
!pip install markdownify

In [4]:
# =============
# Run only once
# The code below:
# Go to demo_gradio, launch section, set share = True
# Go to dots_mocr/model/inference.py, put model_name='rednote-hilab/dots.mocr' in line 41
# =============

import re

# --------- 1. Patch demo_gradio.py (add share=True) ---------
gradio_file = "/content/dots.mocr/demo/demo_gradio.py"

with open(gradio_file, "r") as f:
    code = f.read()

# Replace launch block safely
code = re.sub(
    r"demo\.queue\(\)\.launch\((.*?)\)",
    lambda m: (
        "demo.queue().launch(\n"
        "        server_name=\"0.0.0.0\",\n"
        "        server_port=port,\n"
        "        share=True,\n"
        "        debug=True\n"
        "    )"
    ),
    code,
    flags=re.DOTALL
)

with open(gradio_file, "w") as f:
    f.write(code)

print("✅ Patched demo_gradio.py (share=True added)")


# --------- 2. Patch inference.py (force model name) ---------
infer_file = "/content/dots.mocr/dots_mocr/model/inference.py"

with open(infer_file, "r") as f:
    code = f.read()

# Replace model=model_name → hardcoded model
code = re.sub(
    r"model\s*=\s*model_name",
    'model="rednote-hilab/dots.mocr"',
    code
)

with open(infer_file, "w") as f:
    f.write(code)

print("✅ Patched inference.py (model name fixed)")

✅ Patched demo_gradio.py (share=True added)
✅ Patched inference.py (model name fixed)


Remark: After running the cell below, please wait about 3 to 5 minutes to let the backend start.

To trace the backend, can refer to vllm.log, A "Application startup complete" present in logs means successfully started server.

In [5]:
%env LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia

env: LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia


In [6]:
!CUDA_VISIBLE_DEVICES=0 nohup vllm serve rednote-hilab/dots.mocr \
  --tensor-parallel-size 1 \
  --gpu-memory-utilization 0.7 \
  --chat-template-content-format string \
  --served-model-name rednote-hilab/dots.mocr \
  --trust-remote-code \
  --port 8000 \
  > vllm.log 2>&1 &

There are 2 option to parse documents:
1. Via gradio UI
2. Via inline cmd

## 1. Gradio UI
After running the cell below, a public link will provided, you can interact with the model via the link.

In [ ]:
!python /content/dots.mocr/demo/demo_gradio.py 7860

/content/dots.mocr/demo/demo_gradio.py:701: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme="ocean", css=css, title='dots.mocr') as demo:
/content/dots.mocr/demo/demo_gradio.py:701: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme="ocean", css=css, title='dots.mocr') as demo:
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://4010b35d8b9d6969df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
0.1
use vllm model, num_thread will be set to 64
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File

## 2. Via Inline Cmd

In [13]:
import os

zip_file_path = "/content/table_by_level.zip"
output_dir = "documents"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Unzip the file
!unzip -o "{zip_file_path}" -d "{output_dir}"

print(f"Unzipped {zip_file_path} to {output_dir}")

Archive:  /content/table_by_level.zip
   creating: documents/level_1/
  inflating: documents/level_1/table_0016.jpg  
  inflating: documents/level_1/table_0096.jpg  
  inflating: documents/level_1/table_0101.jpg  
  inflating: documents/level_1/table_0135.jpg  
  inflating: documents/level_1/table_0138.jpg  
   creating: documents/level_2/
  inflating: documents/level_2/table_0079.jpg  
  inflating: documents/level_2/table_0155.jpg  
  inflating: documents/level_2/table_0203.jpg  
  inflating: documents/level_2/table_0281.jpg  
  inflating: documents/level_2/table_0704.jpg  
   creating: documents/level_3/
  inflating: documents/level_3/table_0143.jpg  
  inflating: documents/level_3/table_0331.jpg  
  inflating: documents/level_3/table_0534.jpg  
  inflating: documents/level_3/table_0584.jpg  
  inflating: documents/level_3/table_0804.jpg  
   creating: documents/level_4/
  inflating: documents/level_4/table_0047.jpg  
  inflating: documents/level_4/table_0329.jpg  
  inflating: docum

In [8]:
import sys
sys.path.append("/content/dots.mocr")

from dots_mocr.parser import DotsMOCRParser
from PIL import Image
import os
import time
from datetime import datetime

import threading
import psutil
import subprocess

# --- Global Config ---
INPUT_DIR = "/content/documents"
OUTPUT_DIR = "output"

CSV_LOG_NAME = "log.csv"
JSON_LOG_NAME = "log.json"

parser = DotsMOCRParser(
    ip="127.0.0.1",
    port=8000,
    dpi=200
)

# --- GPU MONITOR ---
def get_gpu_stats():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8")

        util, mem_used, mem_total = result.strip().split(", ")
        return float(util), float(mem_used), float(mem_total)
    except:
        return 0.0, 0.0, 0.0


def get_gpu_temp():
    try:
        temp = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=temperature.gpu",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8").strip()
        return float(temp)
    except:
        return 0.0

def batch_ocr(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    csv_log_file = os.path.join(output_dir, CSV_LOG_NAME)
    json_log_file = os.path.join(output_dir, JSON_LOG_NAME)

    # CSV header
    with open(csv_log_file, "w") as f:
        f.write("filename,status,time_sec,cpu_percent,ram_peak_mb,"
                "gpu_util,gpu_mem,gpu_total,gpu_mem_percent,gpu_temp_peak\n")

    json_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "method": "dots.mocr_parser"
        },
        "records": []
    }

    # files = [
    #     f for f in os.listdir(input_dir)
    #     if f.lower().endswith((".png", ".jpg", ".jpeg"))
    # ]

    files = []

    for root, dirs, filenames in os.walk(INPUT_DIR):
        for f in filenames:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".pdf", ".tiff")):
                full_path = os.path.join(root, f)
                files.append(full_path)

    if not files:
        print("❌ No files found")
        return

    print("=" * 50)
    print("🚀 STARTING BATCH OCR (Parser Mode)")
    print("=" * 50)

    for i, filename in enumerate(files, 1):
        file_path = filename

        relative_path = os.path.relpath(file_path, input_dir)
        name_no_ext = os.path.splitext(relative_path)[0]

        output_path = os.path.join(output_dir, name_no_ext)
        os.makedirs(output_path, exist_ok=True)

        print(f"\n[{i}/{len(files)}] Processing: {filename}")

        # --- MONITOR ---
        cpu_log, ram_log, gpu_log, temp_log = [], [], [], []
        running = True

        start_time = time.time()
        current_pid = os.getpid()

        def monitor():
            while running:
                try:
                    cpu_log.append(psutil.cpu_percent(interval=None))

                    try:
                        ram = psutil.Process(current_pid).memory_info().rss / (1024 ** 2)
                        ram_log.append(ram)
                    except:
                        pass

                    util, mem_used, mem_total = get_gpu_stats()
                    gpu_log.append((util, mem_used, mem_total))

                    temp = get_gpu_temp()
                    if temp:
                        temp_log.append(temp)

                    time.sleep(1)

                except Exception as e:
                    print(f"Monitor error: {e}")
                    break

        thread = threading.Thread(target=monitor, daemon=True)
        thread.start()

        # --- RUN PARSER ---
        try:
            image = Image.open(file_path)

            parser.parse_image(
                input_path=image,
                filename=os.path.basename(name_no_ext),
                prompt_mode="prompt_layout_all_en",
                save_dir=output_path
            )

            status = "SUCCESS"

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            status = "ERROR"

        # --- STOP MONITOR ---
        running = False
        thread.join(timeout=2)

        elapsed = time.time() - start_time

        # --- METRICS ---
        cpu_avg = sum(cpu_log) / len(cpu_log) if cpu_log else 0
        ram_peak = max(ram_log) if ram_log else 0

        if gpu_log:
            gpu_util = sum(x[0] for x in gpu_log) / len(gpu_log)
            gpu_mem = sum(x[1] for x in gpu_log) / len(gpu_log)
            gpu_total = gpu_log[0][2]
            gpu_mem_percent = (gpu_mem / gpu_total * 100) if gpu_total else 0
        else:
            gpu_util = gpu_mem = gpu_total = gpu_mem_percent = 0

        temp_peak = max(temp_log) if temp_log else 0

        # --- PRINT ---
        print(f"✅ {status} | {elapsed:.2f}s")
        print(f"CPU: {cpu_avg:.1f}% | RAM peak: {ram_peak:.0f}MB")
        print(f"GPU: {gpu_util:.1f}% | VRAM: {gpu_mem:.0f}/{gpu_total:.0f}MB ({gpu_mem_percent:.1f}%)")
        if temp_peak:
            print(f"GPU Temp: {temp_peak:.1f}°C")

        # --- CSV LOG ---
        with open(csv_log_file, "a") as f:
            f.write(f"{filename},{status},{elapsed:.2f},{cpu_avg:.1f},{ram_peak:.0f},"
                    f"{gpu_util:.1f},{gpu_mem:.0f},{gpu_total:.0f},{gpu_mem_percent:.1f},{temp_peak:.1f}\n")

        # --- JSON LOG ---
        json_data["records"].append({
            "filename": filename,
            "status": status,
            "time_sec": round(elapsed, 2),
            "cpu_percent": round(cpu_avg, 1),
            "ram_peak_mb": round(ram_peak, 0),
            "gpu": {
                "util": round(gpu_util, 1),
                "mem_used": round(gpu_mem, 0),
                "mem_total": round(gpu_total, 0),
                "mem_percent": round(gpu_mem_percent, 1),
                "temp_peak": round(temp_peak, 1)
            }
        })

    # --- SAVE JSON ---
    with open(json_log_file, "w") as f:
        json.dump(json_data, f, indent=2)

    print("\n🎉 DONE")
    print(f"CSV: {csv_log_file}")
    print(f"JSON: {json_log_file}")


use vllm model, num_thread will be set to 64


In [9]:
import json
import os

# Function to calculate overall summary for the processing time, resources utilization
def calculate_summary(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    records = data.get("records", [])
    if not records:
        print("No records found.")
        return

    total_files = len(records)

    # Accumulators
    total_time = 0
    total_cpu = 0
    total_ram = 0
    total_gpu_util = 0
    total_gpu_mem = 0
    total_gpu_mem_percent = 0
    total_gpu_temp = 0

    for r in records:
        total_time += r["time_sec"]
        total_cpu += r["cpu_percent"]
        total_ram += r["ram_peak_mb"]

        gpu = r.get("gpu", {})
        total_gpu_util += gpu.get("util", 0)
        total_gpu_mem += gpu.get("mem_used", 0)
        total_gpu_mem_percent += gpu.get("mem_percent", 0)
        total_gpu_temp += gpu.get("temp_peak", 0)

    # Averages
    summary = {
        "total_files": total_files,
        "avg_time_sec": total_time / total_files,
        "avg_cpu_percent": total_cpu / total_files,
        "avg_ram_peak_mb": total_ram / total_files,
        "avg_gpu_util": total_gpu_util / total_files,
        "avg_gpu_mem_mb": total_gpu_mem / total_files,
        "avg_gpu_mem_percent": total_gpu_mem_percent / total_files,
        "avg_gpu_temp": total_gpu_temp / total_files
    }

    # ---- SAVE TO TXT ----
    txt_file = os.path.splitext(json_file)[0] + "_summary.txt"

    with open(txt_file, "w") as f:
        f.write("===== OVERALL SUMMARY =====\n")
        for k, v in summary.items():
            if isinstance(v, float):
                f.write(f"{k}: {v:.2f}\n")
            else:
                f.write(f"{k}: {v}\n")

    print(f"✅ Summary saved to: {txt_file}")

    return summary


In [14]:
batch_ocr(INPUT_DIR, OUTPUT_DIR)

JSON_FILE = os.path.join(OUTPUT_DIR, JSON_LOG_NAME)
calculate_summary(JSON_FILE)

🚀 STARTING BATCH OCR (Parser Mode)

[1/20] Processing: /content/documents/level_3/table_0584.jpg
✅ SUCCESS | 9.56s
CPU: 8.7% | RAM peak: 232MB
GPU: 88.9% | VRAM: 13292/23034MB (57.7%)
GPU Temp: 65.0°C

[2/20] Processing: /content/documents/level_3/table_0804.jpg
✅ SUCCESS | 5.31s
CPU: 12.1% | RAM peak: 232MB
GPU: 80.0% | VRAM: 13292/23034MB (57.7%)
GPU Temp: 67.0°C

[3/20] Processing: /content/documents/level_3/table_0331.jpg
✅ SUCCESS | 18.09s
CPU: 9.9% | RAM peak: 232MB
GPU: 94.1% | VRAM: 13292/23034MB (57.7%)
GPU Temp: 71.0°C

[4/20] Processing: /content/documents/level_3/table_0534.jpg
✅ SUCCESS | 14.88s
CPU: 11.8% | RAM peak: 232MB
GPU: 94.1% | VRAM: 13292/23034MB (57.7%)
GPU Temp: 75.0°C

[5/20] Processing: /content/documents/level_3/table_0143.jpg
✅ SUCCESS | 10.62s
CPU: 10.8% | RAM peak: 232MB
GPU: 90.0% | VRAM: 13292/23034MB (57.7%)
GPU Temp: 77.0°C

[6/20] Processing: /content/documents/level_2/table_0281.jpg
✅ SUCCESS | 8.49s
CPU: 9.3% | RAM peak: 232MB
GPU: 87.5% | VRAM: 13

{'total_files': 20,
 'avg_time_sec': 10.248999999999999,
 'avg_cpu_percent': 10.275,
 'avg_ram_peak_mb': 232.0,
 'avg_gpu_util': 85.35,
 'avg_gpu_mem_mb': 13292.0,
 'avg_gpu_mem_percent': 57.700000000000024,
 'avg_gpu_temp': 76.25}

In [15]:
!cd /content && zip -r dots_mocr_output.zip $OUTPUT_DIR

from google.colab import files
files.download("/content/dots_mocr_output.zip")

  adding: output/ (stored 0%)
  adding: output/log.csv (deflated 78%)
  adding: output/level_3/ (stored 0%)
  adding: output/level_3/table_0584/ (stored 0%)
  adding: output/level_3/table_0584/table_0584.jpg (deflated 7%)
  adding: output/level_3/table_0584/table_0584.json (deflated 69%)
  adding: output/level_3/table_0584/table_0584_nohf.md (deflated 71%)
  adding: output/level_3/table_0584/table_0584.md (deflated 71%)
  adding: output/level_3/table_0534/ (stored 0%)
  adding: output/level_3/table_0534/table_0534.jpg (deflated 9%)
  adding: output/level_3/table_0534/table_0534.md (deflated 69%)
  adding: output/level_3/table_0534/table_0534_nohf.md (deflated 69%)
  adding: output/level_3/table_0534/table_0534.json (deflated 68%)
  adding: output/level_3/table_0143/ (stored 0%)
  adding: output/level_3/table_0143/table_0143.json (deflated 70%)
  adding: output/level_3/table_0143/table_0143.md (deflated 72%)
  adding: output/level_3/table_0143/table_0143_nohf.md (deflated 72%)
  adding:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>